# Sentiment and Emotion Prediction Pipeline

This notebook implements a comprehensive pipeline for predicting sentiment and emotions in Reddit posts about homelessness in Canadian cities.

## Pipeline Overview:
1. **Data Loading**: Load Reddit posts from CSV
2. **Text Preprocessing**: Combine title and selftext, clean data
3. **Sentiment Analysis**: Using multiple models
   - Twitter-RoBERTa (sentiment)
   - SiEBERT (RoBERTa-large)
   - BERTweet (social media optimized)
4. **Emotion Classification**: Using multiple models
   - Twitter-RoBERTa (emotion)
   - Twitter-RoBERTa (multi-label)
   - GoEmotions (Reddit-specific)
5. **Results Aggregation**: Combine predictions and save

## 1. Install Required Libraries

In [1]:
# Install required packages
# !pip install transformers accelerate safetensors python-dotenv bitsandbytes torch pandas numpy scipy bertopic sentence-transformers umap-learn hdbscan scikit-learn nbformat


## 2. Import Libraries

In [2]:
import os
import warnings

import pandas as pd
import numpy as np
from collections import Counter
from dotenv import load_dotenv
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM
)
from bertopic import BERTopic
from bertopic.representation import TextGeneration
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import text as sklearn_text
import torch
from scipy.special import softmax

load_dotenv()
warnings.filterwarnings('ignore')


/Users/Kwadwo/Documents/PROJECTS/Homeless_Sentiment/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Load Data

In [3]:
# Load the Reddit posts dataset
df = pd.read_csv('canadian_homelessness_reddit_posts.csv')

print(f"Loaded {len(df)} posts")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

Loaded 2226 posts

Columns: ['id', 'title', 'selftext', 'score', 'num_comments', 'created_utc', 'subreddit', 'url', 'city']

First few rows:


,id,title,selftext,score,num_comments,created_utc,subreddit,url,city
0,1p2wz9z,Experts and activists warn Toronto’s encampmen...,NaN,133,79,2025-11-21 12:10:30,toronto,https://rabble.ca/human-rights/experts-and-act...,Toronto
1,1p28gsh,Solution to homeless in parks and parkettes,"So, I keep seeing them expending tax dollars, ...",0,73,2025-11-20 16:59:50,toronto,https://www.reddit.com/r/toronto/comments/1p28...,Toronto
2,1p21k3z,York Region groups striving to house people wi...,NaN,198,33,2025-11-20 12:11:48,toronto,https://www.durhamregion.com/news/york-region-...,Toronto
3,1ozrfe7,George Hislop Park - fencing and clearing of e...,George Hislop Park - fencing and clearing of e...,139,151,2025-11-17 20:30:39,toronto,https://www.reddit.com/gallery/1ozrfe7,Toronto
4,1ozn8o4,This elderly man stood up to his friend being ...,I was on the subway when this homeless lady wa...,1428,127,2025-11-17 17:58:09,toronto,https://www.reddit.com/r/toronto/comments/1ozn...,Toronto


## 4. Preprocess Text

Combine title and selftext, handle missing values, and truncate long texts

In [4]:
# Create combined text column
df['title'] = df['title'].fillna('')
df['selftext'] = df['selftext'].fillna('')
df['combined_text'] = df['title'] + ' ' + df['selftext']
df['combined_text'] = df['combined_text'].str.strip()

# Remove empty posts
df = df[df['combined_text'] != ''].reset_index(drop=True)

# Truncate very long texts (models have token limits)
# Most models support ~512 tokens, roughly 350-400 words
def truncate_text(text, max_chars=2000):
    if len(text) > max_chars:
        return text[:max_chars] + "..."
    return text

df['combined_text'] = df['combined_text'].apply(truncate_text)

print(f"Preprocessed {len(df)} posts")
print(f"\nSample text:")
print(df['combined_text'].iloc[0][:300])

Preprocessed 2226 posts

Sample text:
Experts and activists warn Toronto’s encampment crackdown masks a deeper housing crisis


### 4.1 Inductive topic modelling

Before running BERTopic, ensure the topic title generator is configured:
- Set `HF_TOKEN` via your shell or `.env` (loaded automatically).
- The first download for `Qwen/Qwen2-7B-Instruct` is ~14 GB.
- Device priority: Apple Metal (MPS) → CUDA GPU → CPU.
- Re-running the loader cell reuses the cached Qwen model; set `force_reload=True` if you need a fresh load.
- Zero-temperature generation keeps topic titles deterministic.


In [5]:
# Determine devices for transformers workloads
if torch.backends.mps.is_available():
    qwen_device = "mps"
    qwen_dtype = torch.float16
elif torch.cuda.is_available():
    qwen_device = "cuda"
    qwen_dtype = torch.float16
else:
    qwen_device = "cpu"
    qwen_dtype = torch.float32

# Device selection for Hugging Face pipelines used later
if torch.cuda.is_available():
    device = 0
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = -1

print(f"Qwen device: {qwen_device} (dtype={qwen_dtype})")
print(f"Pipeline device: {device}")


Qwen device: mps (dtype=torch.float16)
Pipeline device: mps


In [6]:
# Load Qwen 7B model for BERTopic representations
force_reload = False  # Set to True to force a fresh load of the generator
cache_names = ["qwen_tokenizer", "qwen_model", "qwen_generator"]
reuse_cache = (not force_reload) and all(name in globals() for name in cache_names)

hf_token = os.environ.get("HF_TOKEN", "").strip()
if not hf_token:
    raise RuntimeError("HF_TOKEN environment variable is required to load Qwen/Qwen2-7B-Instruct.")

model_id = "Qwen/Qwen2-7B-Instruct"
if reuse_cache:
    qwen_tokenizer = globals()["qwen_tokenizer"]
    qwen_model = globals()["qwen_model"]
    qwen_generator = globals()["qwen_generator"]
    print(f"Reusing cached Qwen model on {qwen_device}.")
else:
    print(f"Loading {model_id} on {qwen_device}...")
    qwen_tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        token=hf_token,
        trust_remote_code=True
    )

    model_kwargs = {
        "torch_dtype": qwen_dtype,
        "trust_remote_code": True
    }

    if qwen_device in {"cuda", "mps"}:
        qwen_model = AutoModelForCausalLM.from_pretrained(
            model_id,
            token=hf_token,
            device_map="auto",
            **model_kwargs
        )
    else:
        qwen_model = AutoModelForCausalLM.from_pretrained(
            model_id,
            token=hf_token,
            **model_kwargs
        )
        qwen_model = qwen_model.to(qwen_device)

    qwen_model.eval()

    qwen_generator = pipeline(
        "text-generation",
        model=qwen_model,
        tokenizer=qwen_tokenizer,
        pad_token_id=qwen_tokenizer.eos_token_id
    )

globals()["qwen_tokenizer"] = qwen_tokenizer
globals()["qwen_model"] = qwen_model
globals()["qwen_generator"] = qwen_generator

cache_status = {
    "reused": reuse_cache,
    "force_reload": force_reload,
    "device": qwen_device
}
print(f"Qwen cache status: {cache_status}")


Loading Qwen/Qwen2-7B-Instruct on mps...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 339/339 [00:09<00:00, 35.47it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Qwen cache status: {'reused': False, 'force_reload': False, 'device': 'mps'}


In [7]:
# Inductive topic modeling with BERTopic
print("Running BERTopic inductive topic modeling on combined text...")

# Configure custom stop words and vectorizer to reduce stop words in topics
custom_stop_words = sorted(
    set(sklearn_text.ENGLISH_STOP_WORDS).union({
        "amp", "https", "reddit"
    })
)
vectorizer_model = CountVectorizer(
    stop_words=custom_stop_words,
    ngram_range=(1, 2),
    min_df=3
)

topic_model = BERTopic(
    language="english",
    calculate_probabilities=True,
    verbose=True,
    vectorizer_model=vectorizer_model,
    low_memory=True
)
topics, probabilities = topic_model.fit_transform(df['combined_text'].tolist())

df['topic_id'] = topics
if probabilities is not None:
    prob_array = np.array(probabilities)
    if prob_array.ndim > 1:
        df['topic_probability'] = prob_array.max(axis=1)
    else:
        df['topic_probability'] = prob_array
else:
    df['topic_probability'] = np.nan

def extract_topic_keywords(topic_id, top_n=5):
    if topic_id == -1:
        return "Outlier"
    topic_terms = topic_model.get_topic(topic_id)
    if not topic_terms:
        return "Outlier"
    return ", ".join(word for word, _ in topic_terms[:top_n])

df['topic_keywords'] = df['topic_id'].apply(extract_topic_keywords)

topic_info = topic_model.get_topic_info()
topic_info['Top_Keywords'] = topic_info['Topic'].apply(lambda topic: extract_topic_keywords(topic, top_n=5))
topic_overview = topic_info

print(topic_info.head(10)[["Topic", "Name", "Top_Keywords", "Count"]])


2026-05-23 12:38:45,938 - BERTopic - Embedding - Transforming documents to embeddings.


Running BERTopic inductive topic modeling on combined text...


Batches: 100%|██████████| 70/70 [00:06<00:00, 11.18it/s]
2026-05-23 12:39:00,407 - BERTopic - Embedding - Completed ✓
2026-05-23 12:39:00,408 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-23 12:39:06,995 - BERTopic - Dimensionality - Completed ✓
2026-05-23 12:39:06,996 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-23 12:39:07,102 - BERTopic - Cluster - Completed ✓
2026-05-23 12:39:07,107 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-23 12:39:07,270 - BERTopic - Representation - Completed ✓


   Topic                                    Name  \
0     -1                  -1_people_like_just_im   
1      0  0_homelessness_homeless_toronto_report   
2      1                   1_man_guy_just_police   
3      2                      2_im_help_work_job   
4      3           3_camping_looking_car_parking   
5      4         4_sleep_doctor_pain_appointment   
6      5              5_music_tent_tents_concert   
7      6                6_noise_loud_night_sleep   
8      7    7_cost_recommendations_rough_looking   
9      8          8_donate_items_clothing_womens   

                                      Top_Keywords  Count  
0                                          Outlier    607  
1  homelessness, homeless, toronto, report, people    138  
2                    man, guy, just, police, didnt    131  
3                   im, help, work, job, resources    117  
4             camping, looking, car, parking, trip    109  
5           sleep, doctor, pain, appointment, help     81  
6      

In [9]:
# Generate reader-friendly topic titles with Qwen
print("Generating reader-friendly topic titles with Qwen...")

if "qwen_generator" not in globals():
    raise RuntimeError("Qwen generator not loaded. Run the Qwen loader cell first.")

title_force_reload = False  # Set to True to regenerate titles even if cached
topic_title_cache = globals().get("qwen_topic_title_cache", {})

def clean_title(prompt: str, generated: str) -> str:
    title = generated.replace(prompt, "").strip()
    if not title:
        return "Untitled"
    title_line = title.split("\n", 1)[0]
    return title_line.strip().strip("\"'")

title_results = {}
for _, row in topic_overview.iterrows():
    topic_id = row["Topic"]
    keywords = row.get("Top_Keywords") or extract_topic_keywords(topic_id, top_n=5)
    if topic_id == -1:
        title_results[topic_id] = "Outlier"
        continue

    cache_key = (topic_id, keywords)
    if not title_force_reload and cache_key in topic_title_cache:
        title_results[topic_id] = topic_title_cache[cache_key]
        continue

    prompt = (
        "You are an expert analyst creating concise, professional topic titles.\n"
        "Craft a title of at most six words that captures the essence of the topic.\n"
        f"Topic keywords: {keywords}\n"
        "Return only the title."
    )
    response = qwen_generator(
        prompt,
        max_new_tokens=32,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=qwen_tokenizer.eos_token_id
    )
    generated = response[0]["generated_text"]
    title = clean_title(prompt, generated)
    topic_title_cache[cache_key] = title
    title_results[topic_id] = title

globals()["qwen_topic_title_cache"] = topic_title_cache

topic_overview["Custom_Title"] = topic_overview["Topic"].map(title_results).fillna("Outlier")
df["custom_topic_title"] = df["topic_id"].map(title_results).fillna("Outlier")

print(topic_overview.head(10)[["Topic", "Custom_Title", "Top_Keywords", "Count"]])

[transformers] Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'pad_token_id', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=32) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generating reader-friendly topic titles with Qwen...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=32) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=32) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=32) and `max_length

   Topic                                       Custom_Title  \
0     -1                                            Outlier   
1      0                Title: "Toronto Homelessness Report   
2      1                Title: "Man Just Released by Police   
3      2                     Seeking Work: Resources & Help   
4      3                     Camping Car Parking Strategies   
5      4                       Doctor's Help for Sleep Pain   
6      5                   Year-Long Music Concert in Tents   
7      6      Title: "Noise and Nighttime Sleep Disturbance   
8      7  Cost-effective Basement Renovation Recommendat...   
9      8             Title: Women's Clothing Donation Drive   

                                      Top_Keywords  Count  
0                                          Outlier    607  
1  homelessness, homeless, toronto, report, people    138  
2                    man, guy, just, police, didnt    131  
3                   im, help, work, job, resources    117  
4     

In [10]:
topic_model.visualize_topics()

In [11]:
topic_info

,Topic,Count,Name,Representation,Representative_Docs,Top_Keywords,Custom_Title
0,-1,607,-1_people_like_just_im,"[people, like, just, im, city, homeless, know,...",[Let’s talk about Living wages I was recently ...,Outlier,Outlier
1,0,138,0_homelessness_homeless_toronto_report,"[homelessness, homeless, toronto, report, peop...",[Best news year to date. I applaud the Victor...,"homelessness, homeless, toronto, report, people","Title: ""Toronto Homelessness Report"
2,1,131,1_man_guy_just_police,"[man, guy, just, police, didnt, said, homeless...",[Followed in the River Valley Not sure how to ...,"man, guy, just, police, didnt","Title: ""Man Just Released by Police"
3,2,117,2_im_help_work_job,"[im, help, work, job, resources, ive, homeless...",[About to be homeless in two weeks! Any soluti...,"im, help, work, job, resources",Seeking Work: Resources & Help
4,3,109,3_camping_looking_car_parking,"[camping, looking, car, parking, trip, park, t...",[Spots for overnight EV charging/sleeping in c...,"camping, looking, car, parking, trip",Camping Car Parking Strategies
5,4,81,4_sleep_doctor_pain_appointment,"[sleep, doctor, pain, appointment, help, surge...",[Shoutout to the JHG Just wanted to share some...,"sleep, doctor, pain, appointment, help",Doctor's Help for Sleep Pain
6,5,80,5_music_tent_tents_concert,"[music, tent, tents, concert, year, going, cal...",[A concert goers review of the 2025 stampede S...,"music, tent, tents, concert, year",Year-Long Music Concert in Tents
7,6,74,6_noise_loud_night_sleep,"[noise, loud, night, sleep, hear, building, la...",[Landlord Making Excessive Noise Late at Night...,"noise, loud, night, sleep, hear","Title: ""Noise and Nighttime Sleep Disturbance"
8,7,69,7_cost_recommendations_rough_looking,"[cost, recommendations, rough, looking, baseme...",[Looking for contractor recommendations for le...,"cost, recommendations, rough, looking, basement",Cost-effective Basement Renovation Recommendat...
9,8,66,8_donate_items_clothing_womens,"[donate, items, clothing, womens, donations, d...",[Best place to donate clothing Looking for the...,"donate, items, clothing, womens, donations",Title: Women's Clothing Donation Drive


In [12]:
topic_info.to_csv('bertopic_topic_info.csv', index=False)

## 5. Sentiment Analysis

### 5.1 Twitter-RoBERTa Sentiment (CardiffNLP)

In [13]:
# Load Twitter-RoBERTa sentiment model
print("Loading Twitter-RoBERTa sentiment model...")
sentiment_twitter_roberta = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=device
)

# Predict sentiment for all posts
def predict_sentiment_roberta(texts, batch_size=16):
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        predictions = sentiment_twitter_roberta(batch, truncation=True, max_length=512)
        results.extend(predictions)
        if (i // batch_size + 1) % 10 == 0:
            print(f"Processed {i+len(batch)}/{len(texts)} posts")
    return results

print("Predicting sentiment with Twitter-RoBERTa...")
sentiment_results_roberta = predict_sentiment_roberta(df['combined_text'].tolist())

# Extract labels and scores
df['sentiment_roberta_label'] = [r['label'] for r in sentiment_results_roberta]
df['sentiment_roberta_score'] = [r['score'] for r in sentiment_results_roberta]

print("\nSentiment distribution (Twitter-RoBERTa):")
print(df['sentiment_roberta_label'].value_counts())

Loading Twitter-RoBERTa sentiment model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 54556.08it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Predicting sentiment with Twitter-RoBERTa...
Processed 160/2226 posts
Processed 320/2226 posts
Processed 480/2226 posts
Processed 640/2226 posts
Processed 800/2226 posts
Processed 960/2226 posts
Processed 1120/2226 posts
Processed 1280/2226 posts
Processed 1440/2226 posts
Processed 1600/2226 posts
Processed 1760/2226 posts
Processed 1920/2226 posts
Processed 2080/2226 posts
Processed 2226/2226 posts

Sentiment distribution (Twitter-RoBERTa):
sentiment_roberta_label
neutral     1231
negative     680
positive     315
Name: count, dtype: int64


### 5.2 SiEBERT (RoBERTa-large for general sentiment)

In [14]:
# Load SiEBERT model (binary: positive/negative)
print("Loading SiEBERT sentiment model...")
sentiment_siebert = pipeline(
    "sentiment-analysis",
    model="siebert/sentiment-roberta-large-english",
    device=device
)

print("Predicting sentiment with SiEBERT...")
sentiment_results_siebert = predict_sentiment_roberta(df['combined_text'].tolist())

df['sentiment_siebert_label'] = [r['label'] for r in sentiment_results_siebert]
df['sentiment_siebert_score'] = [r['score'] for r in sentiment_results_siebert]

print("\nSentiment distribution (SiEBERT):")
print(df['sentiment_siebert_label'].value_counts())

Loading SiEBERT sentiment model...


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 55275.19it/s]


Predicting sentiment with SiEBERT...
Processed 160/2226 posts
Processed 320/2226 posts
Processed 480/2226 posts
Processed 640/2226 posts
Processed 800/2226 posts
Processed 960/2226 posts
Processed 1120/2226 posts
Processed 1280/2226 posts
Processed 1440/2226 posts
Processed 1600/2226 posts
Processed 1760/2226 posts
Processed 1920/2226 posts
Processed 2080/2226 posts
Processed 2226/2226 posts

Sentiment distribution (SiEBERT):
sentiment_siebert_label
neutral     1231
negative     680
positive     315
Name: count, dtype: int64


### 5.3 BERTweet Sentiment (Social Media Optimized)

In [15]:
# Load BERTweet sentiment model
print("Loading BERTweet sentiment model...")
sentiment_bertweet = pipeline(
    "sentiment-analysis",
    model="finiteautomata/bertweet-base-sentiment-analysis",
    device=device
)

print("Predicting sentiment with BERTweet...")
sentiment_results_bertweet = predict_sentiment_roberta(df['combined_text'].tolist())

df['sentiment_bertweet_label'] = [r['label'] for r in sentiment_results_bertweet]
df['sentiment_bertweet_score'] = [r['score'] for r in sentiment_results_bertweet]

print("\nSentiment distribution (BERTweet):")
print(df['sentiment_bertweet_label'].value_counts())

Loading BERTweet sentiment model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 56905.51it/s]
[transformers] emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Predicting sentiment with BERTweet...
Processed 160/2226 posts
Processed 320/2226 posts
Processed 480/2226 posts
Processed 640/2226 posts
Processed 800/2226 posts
Processed 960/2226 posts
Processed 1120/2226 posts
Processed 1280/2226 posts
Processed 1440/2226 posts
Processed 1600/2226 posts
Processed 1760/2226 posts
Processed 1920/2226 posts
Processed 2080/2226 posts
Processed 2226/2226 posts

Sentiment distribution (BERTweet):
sentiment_bertweet_label
neutral     1231
negative     680
positive     315
Name: count, dtype: int64


## 6. Emotion Classification

### 6.1 Twitter-RoBERTa Emotion (Single Label)

In [ ]:
# Load Twitter-RoBERTa emotion model (single-label)
print("Loading Twitter-RoBERTa emotion model...")
emotion_twitter_roberta = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-emotion",
    device=device,
    top_k=None  # Return all emotion scores
)

def predict_emotions(texts, emotion_pipeline, batch_size=16):
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        predictions = emotion_pipeline(batch, truncation=True, max_length=512)
        results.extend(predictions)
        if (i // batch_size + 1) % 10 == 0:
            print(f"Processed {i+len(batch)}/{len(texts)} posts")
    return results

print("Predicting emotions with Twitter-RoBERTa...")
emotion_results_roberta = predict_emotions(df['combined_text'].tolist(), emotion_twitter_roberta)

# Extract primary emotion and all scores
df['emotion_roberta_primary'] = [max(r, key=lambda x: x['score'])['label'] for r in emotion_results_roberta]
df['emotion_roberta_primary_score'] = [max(r, key=lambda x: x['score'])['score'] for r in emotion_results_roberta]

# Store all emotion scores as a dictionary
df['emotion_roberta_all_scores'] = [{e['label']: e['score'] for e in r} for r in emotion_results_roberta]

print("\nEmotion distribution (Twitter-RoBERTa):")
print(df['emotion_roberta_primary'].value_counts())

Loading Twitter-RoBERTa emotion model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 53800.58it/s]


Predicting emotions with Twitter-RoBERTa...
Processed 160/2226 posts
Processed 320/2226 posts
Processed 480/2226 posts
Processed 640/2226 posts
Processed 800/2226 posts
Processed 960/2226 posts
Processed 1120/2226 posts
Processed 1280/2226 posts
Processed 1440/2226 posts
Processed 1600/2226 posts
Processed 1760/2226 posts
Processed 1920/2226 posts
Processed 2080/2226 posts


### 6.2 Twitter-RoBERTa Emotion Multi-Label

In [ ]:
# Load Twitter-RoBERTa multi-label emotion model
print("Loading Twitter-RoBERTa multi-label emotion model...")
emotion_multilabel = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-emotion-multilabel-latest",
    device=device,
    top_k=None
)

print("Predicting emotions with multi-label model...")
emotion_results_multilabel = predict_emotions(df['combined_text'].tolist(), emotion_multilabel)

# For multi-label, store all emotions with score > threshold (e.g., 0.5)
def get_multilabel_emotions(scores, threshold=0.5):
    emotions = [e['label'] for e in scores if e['score'] > threshold]
    return emotions if emotions else ['neutral']

df['emotion_multilabel_labels'] = [get_multilabel_emotions(r) for r in emotion_results_multilabel]
df['emotion_multilabel_all_scores'] = [{e['label']: e['score'] for e in r} for r in emotion_results_multilabel]

# Count top emotions
all_multilabel_emotions = [emotion for emotions in df['emotion_multilabel_labels'] for emotion in emotions]
print("\nTop emotions (Multi-label):")
print(Counter(all_multilabel_emotions).most_common(10))

### 6.3 GoEmotions (Reddit-Specific, 28 Emotions)

In [ ]:
# Load GoEmotions model (Reddit-specific with 28 emotions)
print("Loading GoEmotions model...")
emotion_goemotions = pipeline(
    "text-classification",
    model="SamLowe/roberta-base-go_emotions",
    device=device,
    top_k=None
)

print("Predicting emotions with GoEmotions...")
emotion_results_goemotions = predict_emotions(df['combined_text'].tolist(), emotion_goemotions)

# Get top 3 emotions for each post
df['emotion_goemotions_top3'] = [[e['label'] for e in sorted(r, key=lambda x: x['score'], reverse=True)[:3]] for r in emotion_results_goemotions]
df['emotion_goemotions_all_scores'] = [{e['label']: e['score'] for e in r} for r in emotion_results_goemotions]

# Count top emotions
all_goemotions = [emotion for emotions in df['emotion_goemotions_top3'] for emotion in emotions]
print("\nTop emotions (GoEmotions):")
print(Counter(all_goemotions).most_common(10))

### 6.4 DistilRoBERTa Emotion (General Purpose)

In [ ]:
# Load DistilRoBERTa emotion model (7 emotions: 6 Ekman + neutral)
print("Loading DistilRoBERTa emotion model...")
emotion_distilroberta = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    device=device,
    top_k=None
)

print("Predicting emotions with DistilRoBERTa...")
emotion_results_distilroberta = predict_emotions(df['combined_text'].tolist(), emotion_distilroberta)

df['emotion_distilroberta_primary'] = [max(r, key=lambda x: x['score'])['label'] for r in emotion_results_distilroberta]
df['emotion_distilroberta_primary_score'] = [max(r, key=lambda x: x['score'])['score'] for r in emotion_results_distilroberta]
df['emotion_distilroberta_all_scores'] = [{e['label']: e['score'] for e in r} for r in emotion_results_distilroberta]

print("\nEmotion distribution (DistilRoBERTa):")
print(df['emotion_distilroberta_primary'].value_counts())

## 7. Ensemble and Aggregation

Create consensus predictions from multiple models

In [ ]:
# Map all sentiments to a common scale: negative, neutral, positive
def normalize_sentiment(label):
    label_lower = label.lower()
    if 'neg' in label_lower:
        return 'negative'
    elif 'pos' in label_lower:
        return 'positive'
    else:
        return 'neutral'

df['sentiment_roberta_normalized'] = df['sentiment_roberta_label'].apply(normalize_sentiment)
df['sentiment_siebert_normalized'] = df['sentiment_siebert_label'].apply(normalize_sentiment)
df['sentiment_bertweet_normalized'] = df['sentiment_bertweet_label'].apply(normalize_sentiment)

# Create consensus sentiment (majority vote)
def get_consensus_sentiment(row):
    sentiments = [
        row['sentiment_roberta_normalized'],
        row['sentiment_siebert_normalized'],
        row['sentiment_bertweet_normalized']
    ]
    return Counter(sentiments).most_common(1)[0][0]

df['sentiment_consensus'] = df.apply(get_consensus_sentiment, axis=1)

print("\nConsensus sentiment distribution:")
print(df['sentiment_consensus'].value_counts())
print(f"\nPercentages:")
print(df['sentiment_consensus'].value_counts(normalize=True) * 100)

## 8. Analyze Results by City

In [ ]:
# Sentiment by city
print("\n=== Sentiment by City ===")
city_sentiment = pd.crosstab(df['city'], df['sentiment_consensus'], normalize='index') * 100
print(city_sentiment.round(2))

# Primary emotion by city (using DistilRoBERTa)
print("\n=== Primary Emotion by City (DistilRoBERTa) ===")
city_emotion = pd.crosstab(df['city'], df['emotion_distilroberta_primary'], normalize='index') * 100
print(city_emotion.round(2))

## 9. Visualizations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Plot 1: Overall sentiment distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sentiment distribution
df['sentiment_consensus'].value_counts().plot(kind='bar', ax=axes[0], color=['#d62728', '#7f7f7f', '#2ca02c'])
axes[0].set_title('Consensus Sentiment Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sentiment', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)

# Emotion distribution (DistilRoBERTa)
df['emotion_distilroberta_primary'].value_counts().head(10).plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Top 10 Primary Emotions (DistilRoBERTa)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Emotion', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Sentiment by city
fig, ax = plt.subplots(figsize=(14, 6))
city_sentiment_counts = pd.crosstab(df['city'], df['sentiment_consensus'])
city_sentiment_counts.plot(kind='bar', stacked=True, ax=ax, color=['#d62728', '#7f7f7f', '#2ca02c'])
ax.set_title('Sentiment Distribution by City', fontsize=14, fontweight='bold')
ax.set_xlabel('City', fontsize=12)
ax.set_ylabel('Number of Posts', fontsize=12)
ax.legend(title='Sentiment', title_fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: GoEmotions heatmap by city (top emotions)
# Extract top emotions from GoEmotions
top_emotions_goemotions = Counter(all_goemotions).most_common(10)
top_emotion_labels = [e[0] for e in top_emotions_goemotions]

# Create a matrix of emotion scores by city
emotion_city_matrix = []
cities = df['city'].unique()

for city in cities:
    city_posts = df[df['city'] == city]
    city_emotion_scores = {}
    
    for emotion in top_emotion_labels:
        # Average score for this emotion across all posts in city
        scores = [scores_dict.get(emotion, 0) for scores_dict in city_posts['emotion_goemotions_all_scores']]
        city_emotion_scores[emotion] = np.mean(scores)
    
    emotion_city_matrix.append(city_emotion_scores)

emotion_city_df = pd.DataFrame(emotion_city_matrix, index=cities)

# Plot heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(emotion_city_df, annot=True, fmt='.3f', cmap='YlOrRd', cbar_kws={'label': 'Average Score'})
plt.title('Average Emotion Scores by City (GoEmotions - Top 10)', fontsize=14, fontweight='bold')
plt.xlabel('Emotion', fontsize=12)
plt.ylabel('City', fontsize=12)
plt.tight_layout()
plt.show()

## 10. Save Results

In [ ]:
# Create a simplified version for export (without nested dictionaries)
df_export = df[[
    'id', 'city', 'title', 'selftext', 'combined_text', 'score', 'num_comments', 'created_utc',
    'sentiment_consensus',
    'sentiment_roberta_label', 'sentiment_roberta_score',
    'sentiment_siebert_label', 'sentiment_siebert_score',
    'sentiment_bertweet_label', 'sentiment_bertweet_score',
    'emotion_roberta_primary', 'emotion_roberta_primary_score',
    'emotion_distilroberta_primary', 'emotion_distilroberta_primary_score',
    'url'
]].copy()

# Convert lists to strings for CSV export
df_export['emotion_multilabel_labels'] = df['emotion_multilabel_labels'].apply(lambda x: ', '.join(x))
df_export['emotion_goemotions_top3'] = df['emotion_goemotions_top3'].apply(lambda x: ', '.join(x))

# Save to CSV
df_export.to_csv('reddit_posts_with_sentiment_emotion.csv', index=False)
print("Results saved to 'reddit_posts_with_sentiment_emotion.csv'")

# Save full dataframe with all scores (pickle format to preserve dictionaries)
df.to_pickle('reddit_posts_full_analysis.pkl')
print("Full results (with all scores) saved to 'reddit_posts_full_analysis.pkl'")

## 11. Summary Statistics

In [ ]:
print("="*60)
print("SENTIMENT AND EMOTION ANALYSIS SUMMARY")
print("="*60)

print(f"\nTotal posts analyzed: {len(df)}")
print(f"Cities covered: {df['city'].nunique()}")
print(f"Date range: {df['created_utc'].min()} to {df['created_utc'].max()}")

print("\n" + "="*60)
print("SENTIMENT ANALYSIS")
print("="*60)
print("\nConsensus Sentiment (from 3 models):")
for sentiment, count in df['sentiment_consensus'].value_counts().items():
    pct = count / len(df) * 100
    print(f"  {sentiment.capitalize()}: {count} ({pct:.1f}%)")

print("\n" + "="*60)
print("EMOTION ANALYSIS")
print("="*60)
print("\nTop 5 Primary Emotions (DistilRoBERTa):")
for emotion, count in df['emotion_distilroberta_primary'].value_counts().head(5).items():
    pct = count / len(df) * 100
    print(f"  {emotion.capitalize()}: {count} ({pct:.1f}%)")

print("\nTop 5 Emotions (GoEmotions - Reddit-specific):")
for emotion, count in Counter(all_goemotions).most_common(5):
    print(f"  {emotion.capitalize()}: {count}")

print("\n" + "="*60)
print("MODEL AGREEMENT")
print("="*60)
# Calculate how often all 3 sentiment models agree
agreement = (df['sentiment_roberta_normalized'] == df['sentiment_siebert_normalized']) & \
            (df['sentiment_roberta_normalized'] == df['sentiment_bertweet_normalized'])
agreement_rate = agreement.sum() / len(df) * 100
print(f"\nAll 3 sentiment models agree: {agreement.sum()} posts ({agreement_rate:.1f}%)")

print("\n" + "="*60)

## 12. Sample Predictions

Display detailed predictions for a few sample posts

In [ ]:
# Show detailed predictions for 3 sample posts
sample_indices = df.sample(3, random_state=42).index

for idx in sample_indices:
    print("\n" + "="*80)
    print(f"POST #{idx + 1} - {df.loc[idx, 'city']}")
    print("="*80)
    print(f"\nTitle: {df.loc[idx, 'title']}")
    print(f"\nText preview: {df.loc[idx, 'combined_text'][:300]}...")
    print(f"\nScore: {df.loc[idx, 'score']} | Comments: {df.loc[idx, 'num_comments']}")
    
    print("\n--- SENTIMENT ---")
    print(f"Consensus: {df.loc[idx, 'sentiment_consensus'].upper()}")
    print(f"  Twitter-RoBERTa: {df.loc[idx, 'sentiment_roberta_label']} ({df.loc[idx, 'sentiment_roberta_score']:.3f})")
    print(f"  SiEBERT: {df.loc[idx, 'sentiment_siebert_label']} ({df.loc[idx, 'sentiment_siebert_score']:.3f})")
    print(f"  BERTweet: {df.loc[idx, 'sentiment_bertweet_label']} ({df.loc[idx, 'sentiment_bertweet_score']:.3f})")
    
    print("\n--- EMOTIONS ---")
    print(f"Primary (DistilRoBERTa): {df.loc[idx, 'emotion_distilroberta_primary']} ({df.loc[idx, 'emotion_distilroberta_primary_score']:.3f})")
    print(f"Primary (Twitter-RoBERTa): {df.loc[idx, 'emotion_roberta_primary']} ({df.loc[idx, 'emotion_roberta_primary_score']:.3f})")
    print(f"Multi-label: {', '.join(df.loc[idx, 'emotion_multilabel_labels'])}")
    print(f"GoEmotions top-3: {', '.join(df.loc[idx, 'emotion_goemotions_top3'])}")
    print(f"\nURL: {df.loc[idx, 'url']}")

## Pipeline Complete!

This notebook has successfully:
1. ✅ Loaded Reddit posts about homelessness
2. ✅ Predicted sentiment using 3 state-of-the-art models
3. ✅ Predicted emotions using 4 different models (including Reddit-specific)
4. ✅ Created consensus predictions
5. ✅ Analyzed results by city
6. ✅ Generated visualizations
7. ✅ Saved results to CSV and pickle files

### Files Generated:
- `reddit_posts_with_sentiment_emotion.csv` - Simplified results
- `reddit_posts_full_analysis.pkl` - Complete results with all scores

### Models Used:
**Sentiment:**
- Twitter-RoBERTa (CardiffNLP)
- SiEBERT (RoBERTa-large)
- BERTweet

**Emotion:**
- Twitter-RoBERTa (single-label)
- Twitter-RoBERTa (multi-label)
- GoEmotions (Reddit-specific, 28 emotions)
- DistilRoBERTa (general purpose, 7 emotions)